# Phase 3: Data Cleaning and Transformation

## Step 1: Load Raw Datasets

In [152]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

telco_df = pd.read_excel(
    "../data/telco/Telco_customer_churn.xlsx"
)

retail_churn_df = pd.read_excel(
    "../data/retail_churn/churn_retail.xlsx", sheet_name="E Comm"
)

retail_sales_df = pd.read_csv(
    "../data/retail_sales/retail_data.csv"
)

## Step 2: Create Working Copies

In [153]:
telco_clean = telco_df.copy()

retail_churn_clean = retail_churn_df.copy()

retail_sales_clean = retail_sales_df.copy()

## Step 3: Clean Telco Dataset

### 3.1 Convert Numeric Columns

In [189]:
numeric_cols = [
    "Zip Code",
    "Latitude",
    "Longitude",
    "Monthly Charges",
    "Total Charges",
    "Churn Value",
    "Churn Score",
    "CLTV"
]

for col in numeric_cols:
    telco_clean[col] = pd.to_numeric(
        telco_clean[col],
        errors="coerce"
    )

telco_clean["Total Charges"] = (
    telco_clean["Total Charges"]
    .fillna(
        telco_clean["Total Charges"].median()
    )
)

telco_clean.info()
print()

print(
    telco_clean["Total Charges"]
    .isnull()
    .sum()
)

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 36 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   CustomerID         7043 non-null   str     
 1   Count              7043 non-null   int64   
 2   Country            7043 non-null   str     
 3   State              7043 non-null   str     
 4   City               7043 non-null   str     
 5   Zip Code           7043 non-null   int64   
 6   Lat Long           7043 non-null   str     
 7   Latitude           7043 non-null   float64 
 8   Longitude          7043 non-null   float64 
 9   Gender             7043 non-null   str     
 10  Senior Citizen     7043 non-null   str     
 11  Partner            7043 non-null   str     
 12  Dependents         7043 non-null   str     
 13  Tenure Months      7043 non-null   int64   
 14  Phone Service      7043 non-null   str     
 15  Multiple Lines     7043 non-null   str     
 16  Internet Service 

### 3.2 Handle Missing Values

In [190]:
telco_clean["Tenure_Category"] = pd.cut(
    telco_clean["Tenure Months"],
    bins=[-1, 12, 36, 72],
    labels=[
        "New",
        "Established",
        "Loyal"
    ]
)
telco_clean["Charges_Per_Month"] = np.where(
    telco_clean["Tenure Months"] == 0,
    0,
    telco_clean["Total Charges"] /
    telco_clean["Tenure Months"]
)
telco_clean.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
Tenure_Category         0
Charges_Per_Month       0
Charge_Category         0
dtype: int64

### 3.3 Remove Duplicates

In [191]:
print(
    "Duplicates:",
    telco_clean.duplicated().sum()
)

telco_clean.drop_duplicates(
    inplace=True
)

Duplicates: 0


### 3.4 Clean Text Columns

In [192]:
object_cols = (
    telco_clean
    .select_dtypes(
        include=["object","string"]
    )
    .columns
)

for col in object_cols:
    telco_clean[col] = (
        telco_clean[col]
        .astype(str)
        .str.strip()
    )

## Step 4: Clean Retail Churn Dataset

### Missing Values

In [158]:
# Numeric

# retail_churn_clean.isnull().sum()

numeric_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "NumberOfDeviceRegistered",
    "SatisfactionScore",
    "NumberOfAddress",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount"
]

for col in numeric_cols:
    retail_churn_clean[col] = pd.to_numeric(
        retail_churn_clean[col],
        errors="coerce"
    )

# Impute missing numeric values
for col in numeric_cols:
    retail_churn_clean[col] = (
        retail_churn_clean[col]
        .fillna(
            retail_churn_clean[col].median()
        )
    )

In [159]:
print(
    retail_churn_clean[numeric_cols]
    .isnull()
    .sum()
)

Tenure                         0
WarehouseToHome                0
HourSpendOnApp                 0
NumberOfDeviceRegistered       0
SatisfactionScore              0
NumberOfAddress                0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
CashbackAmount                 0
dtype: int64


In [160]:
# Categorical

cat_cols = (
    retail_churn_clean
    .select_dtypes(
        include=["object","string"]
    )
    .columns
)

for col in cat_cols:
    retail_churn_clean[col] = (
        retail_churn_clean[col]
        .fillna(
            retail_churn_clean[col].mode()[0]
        )
    )

### Remove Duplicates

In [161]:
retail_churn_clean.drop_duplicates(
    inplace=True
)

### Standardize Text

In [162]:
for col in cat_cols:
    retail_churn_clean[col] = (
        retail_churn_clean[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

## Step 5: Clean Retail Sales Dataset

### Missing Values

In [163]:
# Numeric

# retail_sales_clean.isnull().sum()

numeric_cols = (
    retail_sales_clean
    .select_dtypes(
        include=["int64","float64"]
    )
    .columns
)

for col in numeric_cols:

    retail_sales_clean[col] = (
        retail_sales_clean[col]
        .fillna(
            retail_sales_clean[col].median()
        )
    )

In [164]:
# Categorical

cat_cols = (
    retail_sales_clean
    .select_dtypes(
        include=["object","string"]
    )
    .columns
)

for col in cat_cols:

    retail_sales_clean[col] = (
        retail_sales_clean[col]
        .fillna(
            retail_sales_clean[col].mode()[0]
        )
    )

### Remove Duplicates

In [165]:
retail_sales_clean.drop_duplicates(
    inplace=True
)

## Step 6: Date Conversion

In [166]:
date_columns = [
    "transaction_date",
    "last_purchase_date",
    "product_manufacture_date",
    "product_expiry_date",
    "promotion_start_date",
    "promotion_end_date"
]

for col in date_columns:
    retail_sales_clean[col] = pd.to_datetime(
        retail_sales_clean[col],
        errors="coerce"
    )

## Step 7: Feature Engineering

### Telco Features

#### Tenure Months

In [167]:
pd.cut(
    telco_clean["Tenure Months"],
    bins=[0,12,36,72],
    labels=[
        "New",
        "Established",
        "Loyal"
    ],
    include_lowest=True
)

0               New
1               New
2               New
3       Established
4             Loyal
           ...     
7038          Loyal
7039    Established
7040          Loyal
7041            New
7042          Loyal
Name: Tenure Months, Length: 7043, dtype: category
Categories (3, str): ['New' < 'Established' < 'Loyal']

#### Charges Per Month

In [168]:
telco_clean["Charges_Per_Month"] = np.where(
    telco_clean["Tenure Months"] == 0,
    0,
    telco_clean["Total Charges"] /
    telco_clean["Tenure Months"]
)

telco_clean["Charges_Per_Month"].isnull().sum()

np.int64(0)

#### Tenure Category

In [169]:
telco_clean["Tenure_Category"] = pd.cut(
    telco_clean["Tenure Months"],
    bins=[-1, 12, 36, 72],
    labels=[
        "New",
        "Established",
        "Loyal"
    ]
)

telco_clean["Tenure_Category"].isnull().sum()

np.int64(0)

#### Monthly Charge Category

In [170]:
telco_clean["Charge_Category"] = pd.qcut(
    telco_clean["Monthly Charges"],
    q=3,
    labels=["Low", "Medium", "High"]
)
telco_clean["Charge_Category"].isnull().sum()

np.int64(0)

### Retail Churn Features

#### Orders Per Year of Tenure

In [171]:
retail_churn_clean["Orders_Per_Tenure"] = (
    retail_churn_clean["OrderCount"] /
    (retail_churn_clean["Tenure"] + 1)
)

#### Cashback Efficiency

In [172]:
retail_churn_clean["Cashback_Per_Order"] = (
    retail_churn_clean["CashbackAmount"] /
    (retail_churn_clean["OrderCount"] + 1)
)

#### Customer Engagement Score

In [173]:
retail_churn_clean["Engagement_Score"] = (
    retail_churn_clean["HourSpendOnApp"] +
    retail_churn_clean["CouponUsed"] +
    retail_churn_clean["OrderCount"]
)

### Retail Sales Features

#### Online Purchase Ratio

In [174]:
denominator = (
    retail_sales_clean["online_purchases"] +
    retail_sales_clean["in_store_purchases"]
)

retail_sales_clean["online_purchase_ratio"] = np.where(
    denominator == 0,
    0,
    retail_sales_clean["online_purchases"] / denominator
)

#### Return Rate

In [175]:
retail_sales_clean["return_rate"] = (
    retail_sales_clean["total_returned_items"] /
    (
        retail_sales_clean["total_items_purchased"] + 1
    )
)

#### Sales Per Transaction

In [176]:
retail_sales_clean["sales_per_transaction"] = (
    retail_sales_clean["total_sales"] /
    (
        retail_sales_clean["total_transactions"] + 1
    )
)

#### Support Calls Per Purchase

In [177]:
retail_sales_clean["support_calls_per_purchase"] = (
    retail_sales_clean["customer_support_calls"] /
    (
        retail_sales_clean["total_transactions"] + 1
    )
)

#### Customer Value Indicator

In [178]:
retail_sales_clean["customer_value_score"] = (
    retail_sales_clean["total_sales"] *
    (1 - retail_sales_clean["product_return_rate"])
)

## Validation

In [194]:
for name, df in {
    "Telco": telco_clean,
    "Retail Churn": retail_churn_clean,
    "Retail Sales": retail_sales_clean
}.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Shape:", df.shape)

    print(
        "Missing Values:",
        df.isnull().sum().sum()
    )

    print(
        "Duplicate Rows:",
        df.duplicated().sum()
    )

    print(
        "Infinite Values:",
        np.isinf(
            df.select_dtypes(
                include=["int64", "float64"]
            )
        ).sum().sum()
    )

    
    printHeader = True
    for col in df.columns:
        missing = df[col].isnull().sum()
        if missing > 0:
            if printHeader:
                print("\nColumns with Missing Values")
                printHeader = False
            print(
                f"{col}: {missing}"
            )


Telco
Shape: (7043, 36)
Missing Values: 5174
Duplicate Rows: 0
Infinite Values: 0

Columns with Missing Values
Churn Reason: 5174

Retail Churn
Shape: (5630, 23)
Missing Values: 0
Duplicate Rows: 0
Infinite Values: 0

Retail Sales
Shape: (1000000, 83)
Missing Values: 0
Duplicate Rows: 0
Infinite Values: 0


## Create Data Cleaning Summary Report

In [195]:
cleaning_summary = pd.DataFrame({
    "Dataset": [
        "Telco",
        "Retail Churn",
        "Retail Sales"
    ],
    "Rows": [
        len(telco_clean),
        len(retail_churn_clean),
        len(retail_sales_clean)
    ],
    "Columns": [
        len(telco_clean.columns),
        len(retail_churn_clean.columns),
        len(retail_sales_clean.columns)
    ],
    "Missing Values": [
        telco_clean.isnull().sum().sum(),
        retail_churn_clean.isnull().sum().sum(),
        retail_sales_clean.isnull().sum().sum()
    ],
    "Duplicate Rows": [
        telco_clean.duplicated().sum(),
        retail_churn_clean.duplicated().sum(),
        retail_sales_clean.duplicated().sum()
    ]
})

cleaning_summary.to_csv(
    "../reports/data_cleaning_summary.csv",
    index=False
)

## Create Data Cleaning Markdown Report

In [196]:
report_path = Path(
    "../reports/data_cleaning_report.md"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write("# Data Cleaning Report\n\n")

    datasets = [
        ("Telco", telco_clean),
        ("Retail Churn", retail_churn_clean),
        ("Retail Sales", retail_sales_clean)
    ]

    for name, df in datasets:

        f.write(f"## {name}\n\n")

        f.write(
            f"- Rows: {len(df):,}\n"
        )

        f.write(
            f"- Columns: {len(df.columns)}\n"
        )

        f.write(
            f"- Missing Values Remaining: "
            f"{df.isnull().sum().sum():,}\n"
        )

        f.write(
            f"- Duplicate Rows Remaining: "
            f"{df.duplicated().sum():,}\n\n"
        )

        f.write(
            "### Cleaning Actions\n\n"
        )

        f.write(
            "- Missing values treated\n"
        )

        f.write(
            "- Duplicate records removed\n"
        )

        f.write(
            "- Data types standardized\n"
        )

        f.write(
            "- New features created\n\n"
        )

## Create Processed Data Directory and Export Final Datasets

In [197]:
Path(
    "../data/processed"
).mkdir(
    exist_ok=True
)

telco_clean.to_csv(
    "../data/processed/clean_telco.csv",
    index=False
)

retail_churn_clean.to_csv(
    "../data/processed/clean_retail_churn.csv",
    index=False
)

retail_sales_clean.to_csv(
    "../data/processed/clean_retail_sales.csv",
    index=False
)

## Verification of Exported Files

In [198]:
# Reload and verify.
telco_final = pd.read_csv(
    "../data/processed/clean_telco.csv"
)

retail_churn_final = pd.read_csv(
    "../data/processed/clean_retail_churn.csv"
)

retail_sales_final = pd.read_csv(
    "../data/processed/clean_retail_sales.csv"
)

In [199]:
# Check:
print(telco_final.shape)
print(retail_churn_final.shape)
print(retail_sales_final.shape)

(7043, 36)
(5630, 23)
(1000000, 83)


In [200]:
# Compare row counts:
assert len(telco_final) == len(telco_clean)
assert len(retail_churn_final) == len(retail_churn_clean)
assert len(retail_sales_final) == len(retail_sales_clean)

## Create Feature Inventory

In [201]:
feature_inventory = {
    "Telco New Features": [
        "Charges_Per_Month",
        "Tenure_Category",
        "Charge_Category"
    ],
    "Retail Churn New Features": [
        "Orders_Per_Tenure",
        "Cashback_Per_Order",
        "Engagement_Score"
    ],
    "Retail Sales New Features": [
        "online_purchase_ratio",
        "return_rate",
        "sales_per_transaction",
        "support_calls_per_purchase",
        "customer_value_score"
    ]
}

feature_inventory

{'Telco New Features': ['Charges_Per_Month',
  'Tenure_Category',
  'Charge_Category'],
 'Retail Churn New Features': ['Orders_Per_Tenure',
  'Cashback_Per_Order',
  'Engagement_Score'],
 'Retail Sales New Features': ['online_purchase_ratio',
  'return_rate',
  'sales_per_transaction',
  'support_calls_per_purchase',
  'customer_value_score']}

In [202]:
with open(
    "../reports/feature_inventory.json",
    "w"
) as f:
    json.dump(
        feature_inventory,
        f,
        indent=4
    )